# Random Forest Training

In [ ]:
# Pinned to requirements.txt. scikit-learn and librosa matter most: a different
# scikit-learn can fail to unpickle the model in the app, and a different librosa
# can produce different feature values.
!pip install -q scikit-learn==1.9.1 librosa==1.0.0 soundfile==0.14.0 joblib==1.6.0

print()
print("If Colab shows a 'Restart runtime' button above, click it, then carry on")
print("from the Drive mount cell. Do not run this install cell again.")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:

PROJECT_PATH = '/content/drive/MyDrive/SonicSentinel'

In [ ]:
import os
import sys

if not os.path.isdir(PROJECT_PATH):
    raise RuntimeError(
        f"PROJECT_PATH does not exist: {PROJECT_PATH}\n"
        "Check the folder name in your Drive and fix PROJECT_PATH in the cell above."
    )

if not os.path.isfile(os.path.join(PROJECT_PATH, 'feature_extraction', 'features.py')):
    raise RuntimeError(
        f"{PROJECT_PATH} does not look like the project folder.\n"
        "It must contain app.py, config/, feature_extraction/ and src/."
    )

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

try:
    import feature_extraction.features as features_module
    from feature_extraction.features import FEATURE_NAMES, FEATURE_VECTOR_LENGTH
    from config.config import CLASSES, PYTHON_MODELS
    from src.training import run_training
except Exception as error:
    raise RuntimeError(
        "Could not import the project modules from PROJECT_PATH.\n"
        "Check that the whole project folder was uploaded to Drive, including "
        "config/, feature_extraction/, audio_preprocessing/ and src/."
    ) from error


if not os.path.abspath(features_module.__file__).startswith(os.path.abspath(PROJECT_PATH)):
    raise RuntimeError(
        f"features.py was imported from {features_module.__file__}, "
        f"not from {PROJECT_PATH}. Fix sys.path before training."
    )

print("imported the project modules from", PROJECT_PATH)
print("feature module:", features_module.__file__)
print("feature vector length:", FEATURE_VECTOR_LENGTH)
print("classes:", len(CLASSES))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

MODEL_NAME = "random_forest"

estimator = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)


param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

In [ ]:

model = run_training(MODEL_NAME, estimator, param_grid)

In [ ]:
# What this notebook wrote to Drive.
import os

for folder, names in [
    ('python_models', [
        PYTHON_MODELS[MODEL_NAME]['model_file'],
        PYTHON_MODELS[MODEL_NAME]['scaler_file'],
        'label_encoder.pkl',
    ]),
    ('reports', [PYTHON_MODELS[MODEL_NAME]['metrics_file']]),
]:
    for name in names:
        path = os.path.join(PROJECT_PATH, folder, name)
        state = 'saved' if os.path.exists(path) else 'MISSING'
        print(f"{state:8} {folder}/{name}")

print()
print("Download those files from Drive and put them in the same folders in the")
print("project, then set ACTIVE_PYTHON_MODEL in config/config.py to the model")
print("you picked.")

In [ ]:
# The figures the training run wrote, shown here so they can be saved for the
# report. They are also on Drive in reports/plots/.
import glob
import os

from IPython.display import Image, display

pattern = os.path.join(PROJECT_PATH, "reports", "plots", "*.png")
paths = sorted(glob.glob(pattern))

mine = [p for p in paths if os.path.basename(p).startswith(MODEL_NAME)]
shared = [p for p in paths if os.path.basename(p) in
          ("model_comparison.png", "dataset_distribution.png", "example_spectrograms.png")]

if not paths:
    print("No plots found. Did the training cell finish?")

for path in mine + shared:
    print(os.path.basename(path))
    display(Image(filename=path))